# Gamma Recovery — End-to-End Test

Generates 500 mock clusters with a known pivot–magnitude-gap power-law index γ,
saves the truth catalog, runs the destacking likelihood through the nautilus
nested sampler, and plots the recovered posterior against the injected value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

from destacking.simulation import generate_truth_catalog
from destacking.model import make_test_data, run_sampler

## Parameters

In [ ]:
N_CLUSTERS    = 500
TRUE_GAMMA    = 0.5      # the parameter to recover
RADII         = np.logspace(-0.6, 0.6, 8)   # projected radii in Mpc
SEED          = 42

## 1 — Generate truth catalog and save to FITS

In [ ]:
truth = generate_truth_catalog(
    N_CLUSTERS,
    z_range=(0.1, 0.6),
    logM_range=(13.5, 15.0),
    delta_m_gamma=TRUE_GAMMA,
    seed=SEED,
)

truth.write('truth_catalog.fits', overwrite=True)
print(f'Saved {N_CLUSTERS} clusters  |  true γ = {TRUE_GAMMA}')
truth[:5]

In [ ]:
# Quick look at the truth distributions
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

axes[0].hist(truth['z'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Redshift z')

axes[1].hist(np.log10(truth['M_halo']), bins=20, color='steelblue', edgecolor='white')
axes[1].set_xlabel(r'$\log_{10}(M_{\rm halo}\,/\,M_\odot)$')

axes[2].hist(truth['concentration'], bins=20, color='steelblue', edgecolor='white')
axes[2].set_xlabel('Concentration')

dm = truth['delta_m']
axes[3].hist(dm[~np.isnan(dm)], bins=20, color='steelblue', edgecolor='white')
axes[3].set_xlabel(r'Magnitude gap $\Delta m$')

for ax in axes:
    ax.set_ylabel('Count')
plt.suptitle('Truth catalog distributions', y=1.02)
plt.tight_layout()
plt.show()

## 2 — Build synthetic observed data

In [ ]:
cluster_obs, data_vectors, cov_matrices = make_test_data(
    truth,
    RADII,
    fractional_noise=0.2,
    seed=SEED + 1,
)

print(f'Observable table: {cluster_obs.colnames}')
print(f'Data shape:       {data_vectors.shape}')
print(f'Richness range:   {cluster_obs["Lambda"].min():.1f} – {cluster_obs["Lambda"].max():.1f}')
cluster_obs[:5]

In [ ]:
# Sanity check — plot six example profiles
idx = np.round(np.linspace(0, N_CLUSTERS - 1, 6)).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True)

for ax, i in zip(axes.ravel(), idx):
    errs = np.sqrt(np.diag(cov_matrices[i]))
    ax.errorbar(RADII, data_vectors[i], yerr=errs,
                fmt='o', markersize=4, capsize=4, color='steelblue')
    ax.set_xscale('log')
    ax.set_yscale('log')
    dm_str = f'{cluster_obs["delta_m"][i]:.2f}' if np.isfinite(cluster_obs['delta_m'][i]) else 'nan'
    ax.set_title(
        f'z={cluster_obs["z"][i]:.2f}  '
        rf'$\lambda$={cluster_obs["Lambda"][i]:.0f}  '
        rf'$\Delta m$={dm_str}',
        fontsize=9,
    )
    ax.set_xlabel('R [Mpc]')
    ax.set_ylabel(r'$\Delta\Sigma$ [M$_\odot$/pc$^2$]')

plt.suptitle('Sample ΔΣ profiles (20% noise)', y=1.01)
plt.tight_layout()
plt.show()

## 3 — Run the sampler

In [ ]:
sampler = run_sampler(
    cluster_obs,
    data_vectors,
    cov_matrices,
    RADII,
    gamma_range=(0.0, 3.0),
    n_live=500,
    verbose=True,
)

## 4 — Posterior and recovery plot

In [ ]:
points, log_w, log_l = sampler.posterior()
gamma_samples = points[:, 0]
weights       = np.exp(log_w - log_w.max())   # numerically stable normalisation
weights      /= weights.sum()


def weighted_quantile(values, quantiles, weights):
    """Weighted quantiles via sorted cumulative sum."""
    sorter = np.argsort(values)
    v = values[sorter]
    w = weights[sorter]
    cumw = np.cumsum(w) / np.sum(w)
    return np.interp(quantiles, cumw, v)


glo, gmd, ghi = weighted_quantile(gamma_samples, [0.16, 0.50, 0.84], weights)
print(f'True γ  : {TRUE_GAMMA}')
print(f'Recovered: {gmd:.3f}  −{gmd - glo:.3f}  +{ghi - gmd:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.hist(
    gamma_samples, weights=weights, bins=40, density=True,
    color='steelblue', alpha=0.65, edgecolor='white', label='Posterior',
)
ax.axvspan(
    glo, ghi, alpha=0.20, color='steelblue',
    label=rf'68% CI  [{glo:.3f}, {ghi:.3f}]',
)
ax.axvline(gmd,       color='steelblue', lw=1.5, ls='-',
           label=rf'Median $\hat{{\gamma}}$ = {gmd:.3f}')
ax.axvline(TRUE_GAMMA, color='crimson',   lw=2.0, ls='--',
           label=rf'True $\gamma$ = {TRUE_GAMMA}')

ax.set_xlabel(r'$\gamma$  (pivot–magnitude-gap index)', fontsize=13)
ax.set_ylabel('Posterior density', fontsize=12)
ax.set_title(
    rf'Recovery of $\gamma$ from {N_CLUSTERS} clusters  '
    rf'(20\% noise)',
    fontsize=12,
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('gamma_posterior.png', dpi=150)
plt.show()